In [4]:
import numpy as np
import pandas as pd
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import (StratifiedKFold, RandomizedSearchCV,
                                     GridSearchCV, cross_val_score,
                                     train_test_split)
from sklearn.metrics         import classification_report, ConfusionMatrixDisplay
from scipy.stats             import loguniform
import matplotlib.pyplot     as plt
import warnings
warnings.filterwarnings("ignore")

In [5]:
df = pd.read_csv('reduced_features.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27987 entries, 0 to 27986
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Filepath        27987 non-null  object 
 1   Source          27987 non-null  object 
 2   augmented       27987 non-null  int64  
 3   NclCvx_R_mean   27987 non-null  float64
 4   NclCvx_H_mean   27987 non-null  float64
 5   NclCvx_Cr_mean  27987 non-null  float64
 6   NclCvx_B_std    27987 non-null  float64
 7   NclCvx_H_std    27987 non-null  float64
 8   NclCvx_Cr_std   27987 non-null  float64
 9   RocCvx_R_mean   27987 non-null  float64
 10  RocCvx_B_mean   27987 non-null  float64
 11  RocCvx_H_mean   27987 non-null  float64
 12  RocCvx_Cr_mean  27987 non-null  float64
 13  RocCvx_Cb_mean  27987 non-null  float64
 14  RocCvx_G_std    27987 non-null  float64
 15  RocCvx_B_std    27987 non-null  float64
 16  RocCvx_H_std    27987 non-null  float64
 17  RocCvx_A_std    27987 non-null 

In [6]:
LABEL_COL       = "Label"
NON_FEATURE_COLS = ["Filepath", "Source", "augmented", LABEL_COL]
reduced_features = [c for c in df.columns if c not in NON_FEATURE_COLS]

le = LabelEncoder()

# ── train / test split ──────────────────────────────────────────
X_full = df[reduced_features + ["augmented"]].copy()
y_full = pd.Series(
    le.fit_transform(df[LABEL_COL]),   # fit_transform here since le is new
    index=df.index,
    name=LABEL_COL
)

In [8]:
"""
SVM Hyperparameter Search
=========================
Stage 1 — Coarse RandomizedSearchCV  (starting point: C=10, gamma=0.01, rbf)
Stage 2 — Fine GridSearchCV          (tight grid around Stage 1 best)
Stage 3 — Final evaluation           (5-fold CV + held-out test report)

Data  : reduced_features.csv, Original images only
"""

import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.svm                 import SVC
from sklearn.pipeline            import Pipeline
from sklearn.preprocessing       import StandardScaler
from sklearn.model_selection     import (StratifiedKFold, RandomizedSearchCV,
                                         GridSearchCV, cross_val_score,
                                         train_test_split)
from sklearn.metrics             import classification_report, confusion_matrix
from scipy.stats                 import loguniform
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = "reduced_features.csv"
LABEL_COL        = "Label"
NON_FEATURE_COLS = ["Filepath", "Source", "augmented", LABEL_COL]
TEST_SIZE        = 0.20
RANDOM_SEED      = 42
N_JOBS           = -1
CV_FOLDS         = 5

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")

df = df[df["Source"] == "Original"].reset_index(drop=True)
print(f"Original images only: {len(df):,} rows")

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
X = df[feature_cols].values
y = df[LABEL_COL].values
print(f"Features: {len(feature_cols)}  |  X: {X.shape}  |  Classes: {sorted(set(y))}")

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# ── Helpers ───────────────────────────────────────────────────────────────────
def make_pipeline(C=10, gamma=0.01, kernel="rbf"):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svc",    SVC(C=C, gamma=gamma, kernel=kernel,
                       class_weight="balanced",
                       decision_function_shape="ovr",
                       random_state=RANDOM_SEED)),
    ])

def print_section(title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 — Coarse RandomizedSearchCV
# Starting point: C=10, gamma=0.01, kernel=rbf
# ─────────────────────────────────────────────────────────────────────────────
print_section("STAGE 1 — Coarse RandomizedSearchCV")

param_dist = {
    "svc__C":      loguniform(1e-1, 1e4),   # 0.1 → 10,000
    "svc__gamma":  loguniform(1e-5, 1e1),   # 0.00001 → 10
    "svc__kernel": ["rbf", "poly"],
}

rscv = RandomizedSearchCV(
    make_pipeline(),
    param_distributions = param_dist,
    n_iter              = 60,
    cv                  = cv,
    scoring             = "accuracy",
    n_jobs              = N_JOBS,
    random_state        = RANDOM_SEED,
    verbose             = 1,
    refit               = True,
)

t0 = time.time()
rscv.fit(X_train, y_train)
print(f"\n  Best params : {rscv.best_params_}")
print(f"  Best CV acc : {rscv.best_score_*100:.2f}%")
print(f"  Time        : {(time.time()-t0)/60:.1f} min")

results = (pd.DataFrame(rscv.cv_results_)
             .sort_values("mean_test_score", ascending=False)
             .head(10))
print("\n  Top 10:")
for _, r in results.iterrows():
    print(f"    C={r['param_svc__C']:.4f}  gamma={r['param_svc__gamma']:.6f}"
          f"  kernel={r['param_svc__kernel']}  →  {r['mean_test_score']*100:.2f}%")

coarse_best  = rscv.best_params_
coarse_score = rscv.best_score_

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 — Fine GridSearchCV around Stage 1 best
# ─────────────────────────────────────────────────────────────────────────────
print_section("STAGE 2 — Fine GridSearchCV")

def log_grid(center, n_steps=4, factor=2.5):
    lo = center / (factor ** n_steps)
    hi = center * (factor ** n_steps)
    return np.logspace(np.log10(lo), np.log10(hi), 2 * n_steps + 1)

param_grid = {
    "svc__C":      log_grid(coarse_best["svc__C"]),
    "svc__gamma":  log_grid(coarse_best["svc__gamma"]),
    "svc__kernel": [coarse_best["svc__kernel"]],
}

print(f"  Kernel : {coarse_best['svc__kernel']}")
print(f"  C grid : {[f'{v:.4f}' for v in param_grid['svc__C']]}")
print(f"  γ grid : {[f'{v:.6f}' for v in param_grid['svc__gamma']]}")
print(f"  Combos : {len(param_grid['svc__C']) * len(param_grid['svc__gamma'])}")

gscv = GridSearchCV(
    make_pipeline(),
    param_grid = param_grid,
    cv         = cv,
    scoring    = "accuracy",
    n_jobs     = N_JOBS,
    verbose    = 1,
    refit      = True,
)

t0 = time.time()
gscv.fit(X_train, y_train)
print(f"\n  Best params : {gscv.best_params_}")
print(f"  Best CV acc : {gscv.best_score_*100:.2f}%")
print(f"  Time        : {(time.time()-t0)/60:.1f} min")

# Heatmap
pivot = pd.DataFrame(gscv.cv_results_).pivot_table(
    index="param_svc__C", columns="param_svc__gamma", values="mean_test_score"
)
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot * 100, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "CV Accuracy (%)"})
ax.set_title(f"Stage 2 — Fine Grid ({coarse_best['svc__kernel']} kernel)", fontsize=13)
ax.set_xlabel("gamma"); ax.set_ylabel("C")
plt.tight_layout()
plt.savefig("svm_grid_heatmap.png", dpi=150)
plt.close()
print("  Heatmap saved → svm_grid_heatmap.png")

fine_best  = gscv.best_params_
fine_score = gscv.best_score_

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3 — Final evaluation
# ─────────────────────────────────────────────────────────────────────────────
print_section("STAGE 3 — Final Evaluation")

C      = fine_best["svc__C"]
gamma  = fine_best["svc__gamma"]
kernel = fine_best["svc__kernel"]

# 5-fold CV on full training set
pipe   = make_pipeline(C=C, gamma=gamma, kernel=kernel)
scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                         scoring="accuracy", n_jobs=N_JOBS)
print(f"\n  Best params : C={C:.4f}, gamma={gamma:.6f}, kernel={kernel}")
print(f"  CV Accuracy : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")
print(f"  Per-fold    : {[f'{s*100:.1f}%' for s in scores]}")

# Train on full train set → evaluate on held-out test
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(f"\n{'─'*60}")
print("  CLASSIFICATION REPORT (held-out test set)")
print(f"{'─'*60}")
print(classification_report(y_test, y_pred, digits=4))

# Confusion matrices
labels = sorted(set(y_train))
cm     = confusion_matrix(y_test, y_pred, labels=labels)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Confusion Matrix — Counts")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=100)
axes[1].set_title("Confusion Matrix — Row % (recall per class)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.suptitle(f"SVM  C={C:.3f}  γ={gamma:.5f}  kernel={kernel}\n"
             f"CV={scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("svm_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Confusion matrix saved → svm_confusion_matrix.png")

# ── Summary ───────────────────────────────────────────────────────────────────
print_section("SUMMARY")
print(f"  Stage 1 coarse best : {coarse_score*100:.2f}%")
print(f"  Stage 2 fine best   : {fine_score*100:.2f}%")
print(f"  Stage 3 final CV    : {scores.mean()*100:.2f}%")
print(f"\n  Best params:")
for k, v in fine_best.items():
    print(f"    {k} = {v}")
print(f"\n  Outputs: svm_grid_heatmap.png  |  svm_confusion_matrix.png")

Loaded 27,987 rows
Original images only: 17,085 rows
Features: 19  |  X: (17085, 19)  |  Classes: ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
Train: 13,668  |  Test: 3,417

  STAGE 1 — Coarse RandomizedSearchCV
Fitting 5 folds for each of 60 candidates, totalling 300 fits

  Best params : {'svc__C': np.float64(8244.312190905084), 'svc__gamma': np.float64(0.006317967091932507), 'svc__kernel': 'rbf'}
  Best CV acc : 91.93%
  Time        : 6.7 min

  Top 10:
    C=8244.3122  gamma=0.006318  kernel=rbf  →  91.93%
    C=29.4427  gamma=0.013690  kernel=rbf  →  91.08%
    C=3420.9144  gamma=0.000429  kernel=rbf  →  90.22%
    C=65.5206  gamma=0.148969  kernel=rbf  →  90.14%
    C=791.5074  gamma=0.038115  kernel=poly  →  90.00%
    C=1101.5057  gamma=0.000672  kernel=rbf  →  89.99%
    C=4.2259  gamma=0.238582  kernel=poly  →  89.98%
    C=2.3638  gamma=0.023306  kernel=rbf  →  89.58%
    C=2067.8409  gamma=0.054927  kernel=poly  →  89.

In [9]:
"""
SVM Classifier — Best Parameters
=================================
Best params from hyperparameter search:
    C      = 51526.9512
    gamma  = 0.0010108747
    kernel = rbf

Leakage-safe augmented training strategy:
  1. Split originals (augmented==0) into train / test
  2. Add ALL augmented rows (augmented!=0) to the training side only
  3. Test set stays 100% clean originals → zero leakage
"""

import pandas as pd
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = "reduced_features.csv"
LABEL_COL        = "Label"
NON_FEATURE_COLS = ["Filepath", "Source", "augmented", LABEL_COL]
TEST_SIZE        = 0.20
RANDOM_SEED      = 42
CV_FOLDS         = 5

# ── Best params from hyperparameter search ────────────────────────────────────
BEST_C      = 51526.9512
BEST_GAMMA  = 0.0010108747
BEST_KERNEL = "rbf"

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded  : {len(df):,} rows")

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

# ── Step 1: split originals only (augmented == 0) ─────────────────────────────
# Pure originals are split first so the test set is guaranteed clean.
originals = df[df["augmented"] == 0].reset_index(drop=True)
print(f"Originals (augmented==0) : {len(originals):,} rows")

orig_train, orig_test = train_test_split(
    originals,
    test_size    = TEST_SIZE,
    stratify     = originals[LABEL_COL],
    random_state = RANDOM_SEED,
)

# ── Step 2: attach augmented rows to the training side only ───────────────────
# Augmented images (augmented!=0) never enter the test set → no leakage.
augmented = df[df["augmented"] != 0]
print(f"Augmented rows           : {len(augmented):,} rows  (train only)")

train_df = pd.concat([orig_train, augmented], ignore_index=True)

X_train = train_df[feature_cols].values
y_train = train_df[LABEL_COL].values
X_test  = orig_test[feature_cols].values
y_test  = orig_test[LABEL_COL].values

print(f"\nTrain : {len(X_train):,}  (originals: {len(orig_train):,} + augmented: {len(augmented):,})")
print(f"Test  : {len(X_test):,}   (originals only — no leakage)")
print(f"Features : {len(feature_cols)}  |  Classes : {sorted(set(y_train))}")

# ── Build pipeline ────────────────────────────────────────────────────────────
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc",    SVC(C=BEST_C,
                   gamma=BEST_GAMMA,
                   kernel=BEST_KERNEL,
                   class_weight="balanced",
                   decision_function_shape="ovr",
                   random_state=RANDOM_SEED)),
])

# ── 5-fold cross-validation on training set ───────────────────────────────────
cv     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
scores = cross_val_score(svm_pipeline, X_train, y_train,
                         cv=cv, scoring="accuracy", n_jobs=-1)

print(f"\n5-Fold CV Accuracy : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")
print(f"Per-fold           : {[f'{s*100:.1f}%' for s in scores]}")

# ── Train on full train set → evaluate on held-out test set ──────────────────
svm_pipeline.fit(X_train, y_train)
y_pred = svm_pipeline.predict(X_test)

print(f"\n{'─'*60}")
print("CLASSIFICATION REPORT (held-out originals only)")
print(f"{'─'*60}")
print(classification_report(y_test, y_pred, digits=4))

# ── Confusion matrices ────────────────────────────────────────────────────────
labels = sorted(set(y_train))
cm     = confusion_matrix(y_test, y_pred, labels=labels)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Confusion Matrix — Counts")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=100)
axes[1].set_title("Confusion Matrix — Row % (recall per class)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.suptitle(
    f"SVM  |  C={BEST_C}  γ={BEST_GAMMA}  kernel={BEST_KERNEL}\n"
    f"CV Accuracy = {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig("svm_best_model_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Confusion matrix saved → svm_best_model_confusion_matrix.png")

Loaded  : 27,987 rows
Originals (augmented==0) : 10,851 rows
Augmented rows           : 17,136 rows  (train only)

Train : 25,816  (originals: 8,680 + augmented: 17,136)
Test  : 2,171   (originals only — no leakage)
Features : 19  |  Classes : ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

5-Fold CV Accuracy : 93.32% ± 0.23%
Per-fold           : ['93.5%', '93.6%', '93.1%', '93.4%', '93.0%']

────────────────────────────────────────────────────────────
CLASSIFICATION REPORT (held-out originals only)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

    basophil     0.7400    0.9250    0.8222        40
  eosinophil     0.9740    0.9510    0.9624       551
erythroblast     0.8367    0.9111    0.8723        90
          ig     0.9272    0.8917    0.9091       471
  lymphocyte     0.8421    0.9143    0.8767        35
    monocyte     0.6667    0.9091    0.7692        66
 

In [10]:
"""
SVM Classifier — Best Parameters
=================================
Best params from hyperparameter search:
    C      = 51526.9512
    gamma  = 0.0010108747
    kernel = rbf

Leakage-safe augmented training strategy:
  1. Split originals (augmented==0) into train / test
  2. Add ALL augmented rows (augmented!=0) to the training side only
  3. Test set stays 100% clean originals → zero leakage
"""

import numpy as np
import pandas as pd
from sklearn.svm             import SVC
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = "reduced_features.csv"
LABEL_COL        = "Label"
NON_FEATURE_COLS = ["Filepath", "Source", "augmented", LABEL_COL]
TEST_SIZE        = 0.20
RANDOM_SEED      = 42
CV_FOLDS         = 5

# ── Best params from hyperparameter search ────────────────────────────────────
BEST_C      = 51526.9512
BEST_GAMMA  = 0.0010108747
BEST_KERNEL = "rbf"

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded  : {len(df):,} rows")

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

# ── Step 1: split originals only (augmented == 0) ─────────────────────────────
# Pure originals split first — test set is guaranteed clean.
# Per-class minimum: each class contributes at least MIN_TEST_PER_CLASS samples
# to the test set, even if that exceeds 20% for rare classes.
originals         = df[df["augmented"] == 0].reset_index(drop=True)
MIN_TEST_PER_CLASS = 100
print(f"Originals (augmented==0)  : {len(originals):,} rows")

rng         = np.random.default_rng(RANDOM_SEED)
train_parts = []
test_parts  = []

for label, grp in originals.groupby(LABEL_COL):
    grp       = grp.sample(frac=1, random_state=RANDOM_SEED)   # shuffle
    n_test    = max(int(len(grp) * TEST_SIZE), MIN_TEST_PER_CLASS)
    n_test    = min(n_test, len(grp))                           # can't exceed class size
    test_parts.append(grp.iloc[:n_test])
    train_parts.append(grp.iloc[n_test:])
    print(f"  {label:<15} total={len(grp):>4}  test={n_test:>4}  train={len(grp)-n_test:>4}")

orig_train = pd.concat(train_parts, ignore_index=True)
orig_test  = pd.concat(test_parts,  ignore_index=True)
print(f"\nOrig train : {len(orig_train):,}  |  Orig test : {len(orig_test):,}")

# ── Step 2: attach augmented rows to the training side only ───────────────────
# Augmented images (augmented!=0) never enter the test set → no leakage.
augmented = df[df["augmented"] != 0]
print(f"Augmented rows           : {len(augmented):,} rows  (train only)")

train_df = pd.concat([orig_train, augmented], ignore_index=True)

X_train = train_df[feature_cols].values
y_train = train_df[LABEL_COL].values
X_test  = orig_test[feature_cols].values
y_test  = orig_test[LABEL_COL].values

print(f"\nTrain : {len(X_train):,}  (originals: {len(orig_train):,} + augmented: {len(augmented):,})")
print(f"Test  : {len(X_test):,}   (originals only — no leakage)")
print(f"Features : {len(feature_cols)}  |  Classes : {sorted(set(y_train))}")

# ── Build pipeline ────────────────────────────────────────────────────────────
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc",    SVC(C=BEST_C,
                   gamma=BEST_GAMMA,
                   kernel=BEST_KERNEL,
                   class_weight="balanced",
                   decision_function_shape="ovr",
                   random_state=RANDOM_SEED)),
])

# ── 5-fold cross-validation on training set ───────────────────────────────────
cv     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
scores = cross_val_score(svm_pipeline, X_train, y_train,
                         cv=cv, scoring="accuracy", n_jobs=-1)

print(f"\n5-Fold CV Accuracy : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")
print(f"Per-fold           : {[f'{s*100:.1f}%' for s in scores]}")

# ── Train on full train set → evaluate on held-out test set ──────────────────
svm_pipeline.fit(X_train, y_train)
y_pred = svm_pipeline.predict(X_test)

print(f"\n{'─'*60}")
print("CLASSIFICATION REPORT (held-out originals only)")
print(f"{'─'*60}")
print(classification_report(y_test, y_pred, digits=4))

# ── Confusion matrices ────────────────────────────────────────────────────────
labels = sorted(set(y_train))
cm     = confusion_matrix(y_test, y_pred, labels=labels)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("Confusion Matrix — Counts")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[1], vmin=0, vmax=100)
axes[1].set_title("Confusion Matrix — Row % (recall per class)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.suptitle(
    f"SVM  |  C={BEST_C}  γ={BEST_GAMMA}  kernel={BEST_KERNEL}\n"
    f"CV Accuracy = {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig("svm_best_model_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Confusion matrix saved → svm_best_model_confusion_matrix.png")

Loaded  : 27,987 rows
Originals (augmented==0)  : 10,851 rows
  basophil        total= 202  test= 100  train= 102
  eosinophil      total=2753  test= 550  train=2203
  erythroblast    total= 450  test= 100  train= 350
  ig              total=2354  test= 470  train=1884
  lymphocyte      total= 176  test= 100  train=  76
  monocyte        total= 328  test= 100  train= 228
  neutrophil      total=3166  test= 633  train=2533
  platelet        total=1422  test= 284  train=1138

Orig train : 8,514  |  Orig test : 2,337
Augmented rows           : 17,136 rows  (train only)

Train : 25,650  (originals: 8,514 + augmented: 17,136)
Test  : 2,337   (originals only — no leakage)
Features : 19  |  Classes : ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']

5-Fold CV Accuracy : 93.25% ± 0.43%
Per-fold           : ['93.6%', '93.1%', '93.9%', '92.9%', '92.7%']

────────────────────────────────────────────────────────────
CLASSIFICATION REPORT (held-ou